In [1]:
import ast
import json
from pathlib import Path

import numpy as np
import pandas as pd


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

In [2]:
train_poc = pd.read_csv(
    DATA_DIR / "train_poc.csv",
    usecols=["id", "input_text", "true_risks"]
)

train_input = pd.read_csv(
    OUTPUT_DIR / "stage_c_train_input.csv"
)

test_input = pd.read_csv(
    OUTPUT_DIR / "stage_c_test_input.csv"
)


train_poc["true_risks"] = (
    train_poc["true_risks"].apply(ast.literal_eval)
)

In [3]:
json_columns = [
    "stage_a_predictions",
    "stage_b_predictions",
    "accepted_predictions",
    "stage_b_probabilities"
]


for dataframe in [train_input, test_input]:

    for column in json_columns:
        dataframe[column] = dataframe[column].apply(
            json.loads
        )

    dataframe["route_to_stage_c"] = (
        dataframe["route_to_stage_c"]
        .astype(str)
        .str.lower()
        .eq("true")
    )

In [4]:
saved_embeddings = np.load(
    OUTPUT_DIR / "stage_b_embeddings.npz",
    allow_pickle=False
)

train_ids = saved_embeddings["train_ids"]
test_ids = saved_embeddings["test_ids"]

train_embeddings = saved_embeddings["train_embeddings"]
test_embeddings = saved_embeddings["test_embeddings"]


assert np.array_equal(
    train_ids,
    train_poc["id"].to_numpy()
)

print(
    "Embedding model:",
    saved_embeddings["model_name"].item()
)

Embedding model: sentence-transformers/all-MiniLM-L6-v2


In [5]:
routed_train = train_input[
    train_input["route_to_stage_c"]
].copy()

routed_test = test_input[
    test_input["route_to_stage_c"]
].copy()


sample_sizes = {
    "predictions_disagree": 45,
    "both_stages_empty": 40,
    "stage_b_empty": 15
}


training_samples = []

for reason, sample_size in sample_sizes.items():

    group = routed_train[
        routed_train["stage_c_route_reason"] == reason
    ]

    training_samples.append(
        group.sample(
            n=min(sample_size, len(group)),
            random_state=42
        )
    )


train_llm_input = pd.concat(
    training_samples,
    ignore_index=True
)

test_llm_input = routed_test.reset_index(
    drop=True
)


print("Training LLM records:", len(train_llm_input))
print("Test LLM records:", len(test_llm_input))

Training LLM records: 100
Test LLM records: 194


In [6]:
train_position = {
    event_id: position
    for position, event_id in enumerate(train_ids)
}

test_position = {
    event_id: position
    for position, event_id in enumerate(test_ids)
}


train_query_embeddings = train_embeddings[
    [
        train_position[event_id]
        for event_id in train_llm_input["id"]
    ]
]

test_query_embeddings = test_embeddings[
    [
        test_position[event_id]
        for event_id in test_llm_input["id"]
    ]
]

In [7]:
train_lookup = train_poc.set_index("id")


def add_retrieved_examples(
    queries,
    query_embeddings,
    exclude_self=False,
    k=3
):

    similarities = (
        query_embeddings
        @ train_embeddings.T
    )

    if exclude_self:

        for row_number, event_id in enumerate(
            queries["id"]
        ):
            similarities[
                row_number,
                train_position[event_id]
            ] = -np.inf

    nearest_positions = np.argsort(
        similarities,
        axis=1
    )[:, -k:][:, ::-1]

    output = queries.copy()

    output["retrieved_examples"] = [
        [
            {
                "id": int(train_ids[position]),
                "text": train_lookup.at[
                    int(train_ids[position]),
                    "input_text"
                ],
                "true_risks": train_lookup.at[
                    int(train_ids[position]),
                    "true_risks"
                ],
                "similarity": round(
                    float(similarities[row_number, position]),
                    4
                )
            }
            for position in positions
        ]
        for row_number, positions in enumerate(
            nearest_positions
        )
    ]

    return output

In [8]:
train_llm_input = add_retrieved_examples(
    train_llm_input,
    train_query_embeddings,
    exclude_self=True
)

test_llm_input = add_retrieved_examples(
    test_llm_input,
    test_query_embeddings
)


train_llm_input[
    [
        "id",
        "stage_c_route_reason",
        "retrieved_examples"
    ]
].head()

,id,stage_c_route_reason,retrieved_examples
0,2766,predictions_disagree,"[{'id': 2765, 'text': 'UPDATE: Average waiting..."
1,4557,predictions_disagree,"[{'id': 3872, 'text': 'Operations at Port of S..."
2,2574,predictions_disagree,"[{'id': 1718, 'text': 'Japan: Around 1.8 milli..."
3,2911,predictions_disagree,"[{'id': 2830, 'text': 'UPDATE: Intermittent po..."
4,38,predictions_disagree,"[{'id': 798, 'text': 'UPDATE - USA, New York: ..."


In [9]:
%pip install -q openai

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import getpass

from openai import OpenAI


api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    api_key = getpass.getpass(
        "Enter OpenAI API key: "
    )

client = OpenAI(api_key=api_key)

LLM_MODEL = "gpt-4.1-mini"

In [ ]:
# Taxonomy and output format
RISK_TAXONOMY = {
    "weather_disruption":
        "Disruption caused by severe weather.",

    "natural_disaster":
        "Disruption caused by earthquakes, tsunamis, volcanoes or landslides.",

    "port_operational_disruption":
        "Port congestion, delays, backlogs or reduced operational capacity.",

    "port_closure":
        "Full or partial closure or suspension of a port or terminal.",

    "labor_strike_disruption":
        "Disruption caused by strikes or other labor action.",

    "maritime_security_navigation_disruption":
        "Piracy, maritime security threats, navigation restrictions or waterway disruption."
}


RISK_IDS = list(RISK_TAXONOMY)


DECISION_FORMAT = {
    "type": "json_schema",
    "name": "stage_c_decision",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "predicted_risks": {
                "type": "array",
                "items": {
                    "type": "string",
                    "enum": RISK_IDS
                }
            },
            "reason": {
                "type": "string"
            }
        },
        "required": [
            "predicted_risks",
            "reason"
        ],
        "additionalProperties": False
    }
}

In [ ]:
# Prompt, adjudication and fallback
INSTRUCTIONS = f"""
You are the final adjudicator for a multi-label risk classifier.

Risk taxonomy:
{json.dumps(RISK_TAXONOMY, indent=2)}

Use the event text as the main evidence.
Stage A, Stage B and the retrieved examples are supporting evidence.
Select only risks directly supported by the target event.
Keep the reason brief.
"""


def simple_fallback(row):

    if row["stage_b_predictions"]:
        return row["stage_b_predictions"]

    if row["stage_a_predictions"]:
        return row["stage_a_predictions"]

    return []


def adjudicate(row):

    case = {
        "event_text": row["input_text"],
        "stage_a_predictions":
            row["stage_a_predictions"],
        "stage_b_predictions":
            row["stage_b_predictions"],
        "stage_b_probabilities":
            row["stage_b_probabilities"],
        "route_reason":
            row["stage_c_route_reason"],
        "retrieved_examples":
            row["retrieved_examples"]
    }

    last_error = ""

    for _ in range(2):

        try:
            response = client.responses.create(
                model=LLM_MODEL,
                instructions=INSTRUCTIONS,
                input=json.dumps(
                    case,
                    ensure_ascii=False
                ),
                text={
                    "format": DECISION_FORMAT
                },
                temperature=0,
                max_output_tokens=200,
                store=False
            )

            result = json.loads(
                response.output_text
            )

            predictions = list(
                dict.fromkeys(
                    result["predicted_risks"]
                )
            )

            return (
                predictions,
                result["reason"],
                False
            )

        except Exception as error:
            last_error = str(error)

    return (
        simple_fallback(row),
        f"Fallback used: {last_error[:150]}",
        True
    )


In [ ]:
# Run adjudication
def run_adjudication(dataframe):

    output = dataframe.copy().reset_index(
        drop=True
    )

    results = []

    for number, (_, row) in enumerate(
        output.iterrows(),
        start=1
    ):

        results.append(
            adjudicate(row)
        )

        if number % 10 == 0:
            print(
                f"Completed {number} of {len(output)}"
            )

    output[
        [
            "stage_c_predictions",
            "stage_c_reason",
            "fallback_used"
        ]
    ] = pd.DataFrame(
        results,
        index=output.index
    )

    return output

In [ ]:
# Evaluate the training sample
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer


def evaluate_predictions(
    dataframe,
    prediction_column
):

    label_binarizer = MultiLabelBinarizer(
        classes=RISK_IDS
    )

    true_matrix = label_binarizer.fit_transform(
        dataframe["true_risks"]
    )

    predicted_matrix = label_binarizer.transform(
        dataframe[prediction_column]
    )

    print(
        classification_report(
            true_matrix,
            predicted_matrix,
            target_names=RISK_IDS,
            zero_division=0
        )
    )

    return {
        "micro_f1": f1_score(
            true_matrix,
            predicted_matrix,
            average="micro",
            zero_division=0
        ),
        "macro_f1": f1_score(
            true_matrix,
            predicted_matrix,
            average="macro",
            zero_division=0
        ),
        "exact_match": np.mean(
            np.all(
                true_matrix == predicted_matrix,
                axis=1
            )
        )
    }

In [ ]:
train_evaluation = train_llm_input.merge(
    train_poc[
        ["id", "true_risks"]
    ],
    on="id",
    how="left",
    validate="one_to_one"
)


train_sample_metrics = evaluate_predictions(
    train_evaluation,
    "stage_c_predictions"
)

train_sample_metrics

In [ ]:
# Run the test adjudication
test_llm_input = run_adjudication(
    test_llm_input
)

In [ ]:
# Build final test predictions

test_llm_input["retrieved_ids"] = (
    test_llm_input["retrieved_examples"].apply(
        lambda examples: [
            example["id"]
            for example in examples
        ]
    )
)


stage_c_results = test_llm_input[
    [
        "id",
        "stage_c_predictions",
        "stage_c_reason",
        "fallback_used",
        "retrieved_ids"
    ]
]


test_final = test_input.merge(
    stage_c_results,
    on="id",
    how="left",
    validate="one_to_one"
)


test_final["final_predictions"] = [
    stage_c_predictions
    if route_to_stage_c
    else accepted_predictions

    for route_to_stage_c,
        stage_c_predictions,
        accepted_predictions
    in zip(
        test_final["route_to_stage_c"],
        test_final["stage_c_predictions"],
        test_final["accepted_predictions"]
    )
]


test_final["fallback_used"] = (
    test_final["fallback_used"]
    .fillna(False)
    .astype(bool)
)

In [ ]:
# Final evaluation and export
test_truth = pd.read_csv(
    DATA_DIR / "test_poc.csv",
    usecols=["id", "true_risks"]
)

test_truth["true_risks"] = (
    test_truth["true_risks"].apply(
        ast.literal_eval
    )
)


test_evaluation = test_final.merge(
    test_truth,
    on="id",
    how="left",
    validate="one_to_one"
)


full_test_metrics = evaluate_predictions(
    test_evaluation,
    "final_predictions"
)

routed_test_metrics = evaluate_predictions(
    test_evaluation[
        test_evaluation["route_to_stage_c"]
    ],
    "final_predictions"
)

In [ ]:
metrics = pd.DataFrame(
    [
        train_sample_metrics,
        full_test_metrics,
        routed_test_metrics
    ],
    index=[
        "training_sample",
        "full_test_pipeline",
        "routed_test_only"
    ]
)


test_export = test_final.copy()

json_columns = [
    "stage_a_predictions",
    "stage_b_predictions",
    "stage_b_probabilities",
    "accepted_predictions",
    "stage_c_predictions",
    "retrieved_ids",
    "final_predictions"
]


for column in json_columns:

    test_export[column] = test_export[column].apply(
        lambda value: json.dumps(
            value,
            ensure_ascii=False
        )
        if isinstance(value, (list, dict))
        else value
    )


train_evaluation.to_csv(
    OUTPUT_DIR /
    "stage_c_train_sample_predictions.csv",
    index=False
)

test_export.to_csv(
    OUTPUT_DIR /
    "stage_c_test_predictions.csv",
    index=False
)

metrics.to_csv(
    OUTPUT_DIR / "stage_c_metrics.csv"
)


print("Stage C outputs saved.")